# 06. Year-to-year balance attribution

Attribute every annual change in the General Government balance to changes in the three subsectors, and distinguish the level of a balance from its contribution to an annual movement.

**Reads**

- `outputs/tables/balance_change_attribution.csv`
- `outputs/tables/largest_balance_movements.csv`

**Writes**

- Nothing. Both tables are persisted by the pipeline.

**Method reference:** `METHODOLOGY.md` section 6

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

%matplotlib inline

from portugal_fiscal_balance.analysis import figures

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. An exact decomposition

$$\Delta B^{GG}_t = \Delta B^{C}_t + \Delta B^{RL}_t + \Delta B^{SSF}_t.$$

This holds by construction, so the table is a reallocation of an observed change
rather than an estimate.

The same change is also reported scaled by current-year GDP. Because the three
terms then share one denominator, the scaled version decomposes exactly as well:

$$\frac{\Delta B^{GG}_t}{GDP_t} = \frac{\Delta B^{C}_t}{GDP_t} + \frac{\Delta B^{RL}_t}{GDP_t} + \frac{\Delta B^{SSF}_t}{GDP_t}.$$

That is **not** the change in the balance ratio, which would also move with the
denominator and would not decompose additively. The scaling exists so that years
can be compared in size: ranking movements on nominal euro effectively ranks them
by how recent they are.

In [ ]:
attribution = pd.read_csv(TABLES / 'balance_change_attribution.csv')
change_columns = [
    'year',
    'aggregate_change_m_eur',
    'central_change_m_eur',
    'regional_local_change_m_eur',
    'ssf_change_m_eur',
    'aggregate_change_pct_gdp',
    'change_closure_error_m_eur',
]
display(attribution[change_columns].tail(12).round(3))

## 2. Both windows, drawn separately

The attribution is plotted as two panels that stop either side of 1995. A single
panel spanning the splice would place a vintage revision among the economic
movements and give it the same visual weight. The window is written into each
title, because a stacked bar chart gives the reader no other way to tell that
years are missing from the ends.

In [ ]:
figure = figures.balance_change_attribution(attribution, start_year=1996, end_year=2025)

In [ ]:
figure = figures.balance_change_attribution(attribution, start_year=1978, end_year=1994)

## 3. Contribution shares

Shares are expressed against the absolute aggregate change, so a share above one
means a subsector moved further than the aggregate and was partly offset by
another subsector. A negative share means the subsector moved against the
aggregate.

In [ ]:
share_columns = [
    'year',
    'aggregate_change_m_eur',
    'central_change_share_abs_aggregate_change',
    'regional_local_change_share_abs_aggregate_change',
    'ssf_change_share_abs_aggregate_change',
]
largest = attribution.reindex(attribution['aggregate_change_m_eur'].abs().sort_values(ascending=False).index)
display(largest[share_columns].head(10).round(3))

## 4. The largest movements, ranked inside each regime

Ranked on the GDP-scaled change, which removes the recency bias of a nominal
ranking, and **within** each statistical regime rather than across both. Each annual
change is computed inside one source family and is sound, but ordering historical
against modern episodes by size would compare two methodologies -- the thing this
analysis refuses to do with magnitudes everywhere else.

1995 is excluded because that change straddles the vintage splice in both panels.

In [ ]:
movements = pd.read_csv(TABLES / 'largest_balance_movements.csv')
display(
    movements[
        [
            'regime',
            'rank_in_regime',
            'year',
            'direction',
            'aggregate_change_pct_gdp',
            'aggregate_change_m_eur',
            'dominant_subsector',
            'dominant_subsector_share',
        ]
    ].round(3)
)

The attribution is hierarchical: the revenue and expenditure split below is of the
subsector that dominates each move, not of the aggregate, so both halves describe the
same entity. Expenditure enters the balance negatively, so its contribution is minus
the change.

In [ ]:
display(
    movements[
        [
            'regime',
            'year',
            'dominant_subsector',
            'dominant_subsector_change_m_eur',
            'dominant_revenue_change_m_eur',
            'dominant_expenditure_contribution_m_eur',
            'dominant_split_error_m_eur',
        ]
    ].round(1)
)

## 5. Closure check

In [ ]:
print('max |change-identity residual| (M EUR):', round(float(attribution['change_closure_error_m_eur'].abs().max()), 6))
print('observations:', len(attribution), 'covering', int(attribution['year'].min()), 'to', int(attribution['year'].max()))

## Interpretation limits

1. **Contribution is not causation.** A subsector accounting for most of an
   annual improvement has not been shown to have produced it.
2. The 1994-to-1995 change crosses the **statistical splice** and mixes a
   vintage revision with an economic movement. It is excluded from the ranked
   episodes and from both figures.
3. **The GDP-scaled change is not the change in the balance ratio.** It shares a
   denominator so the decomposition stays exact.
4. Attribution operates on **balances only**. Whether a movement came from
   revenue or expenditure is the subject of notebook 05.

---

[Previous: 05. Revenue and expenditure decomposition](05_revenue_expenditure.ipynb) | [Next: 07. Balance persistence and sign transitions](07_persistence.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```